# 유튜브 영상 다운로드 + video-to-text

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
youtube_api_key = os.getenv("YOUTUBE_API_KEY");
gemini_api_key = os.getenv("GEMINI_API_KEY");

### 유튜브 다운로드로 분석

##### 유튜브 검색, 타이틀id 가져오기

In [ ]:
from googleapiclient.discovery import build

youtube = build("youtube", "v3", developerKey=youtube_api_key)

print("골반 통신 search().list() ")

video_response = youtube.search().list(
    part="snippet",
    q="골반 통신",            # 검색 키워드
    type="video",            # 영상만
    regionCode="KR",
    videoDuration = "short", # 4분 미만만
    videoDefinition="high",  # 세로
    maxResults=1
).execute()

if not video_response["items"]:
    print("영상을 찾을 수 없습니다.")
    exit()

top_video = video_response["items"][0]
target_video_id = top_video["id"]["videoId"] 
target_video_title = top_video["snippet"]["title"]

print(f"제목 : {target_video_title}")
print(f"영상 ID: {target_video_id}")
print("-" * 40)

골반 통신 search().list() 
제목 : 골반통신
영상 ID: 9JzNU0hrR0E
----------------------------------------


##### 영상 길이 확인

In [3]:
video_info = youtube.videos().list(
    part="contentDetails,statistics",
    id=target_video_id
).execute()

# 영상 길이 (ISO 8601 형식)
duration = video_info["items"][0]["contentDetails"]["duration"]
print(f"길이: {duration}")  # 예: PT3M45S (3분 45초)

길이: PT53S


##### 영상 다운로드(yt-dlp)

In [4]:
# pip install yt-dlp

In [19]:
import yt_dlp

def download_youtube_video(video_title, video_id):
    ydl_opts = {
        'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',  # 최고 화질
        'outtmpl': f'../../data/raw/videos/{video_id}.mp4',  # 저장 경로
        'quiet': True,  # 로그 최소화
        'js_runtimes': {
            'node': {}
        },
        
    }
    
    # URL 생성
    url = f'https://www.youtube.com/watch?v={video_id}'
    
    # 다운로드 실행
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        # 정보 추출 (다운로드 포함)
        info = ydl.extract_info(url, download=True)
        
        # 파일 크기 출력 (바이트 → MB)
        filesize = info.get('filesize') or info.get('filesize_approx')
        if filesize:
            print(f"파일 크기: {filesize / 1024 / 1024:.2f}MB")
        else:
            print("파일 크기 정보 없음")
    
    # 다운로드된 파일 경로 반환
    return f'../../data/raw/videos/{video_id}.mp4'

# 사용 예시
video_path = download_youtube_video(target_video_title, target_video_id)
print(f"다운로드 완료: {video_path}")

파일 크기: 3.43MB                                              
다운로드 완료: ../../data/raw/videos/9JzNU0hrR0E.mp4


##### Gemini 2.5-Flash (API로 다운받은 영상보내기)

In [ ]:
# pip install google-genai


  Attempting uninstall: google-auth

    Found existing installation: google-auth 2.43.0

    Uninstalling google-auth-2.43.0:

      Successfully uninstalled google-auth-2.43.0

   ---------------------------------------- 0/2 [google-auth]
   ---------------------------------------- 0/2 [google-auth]
   -------------------- ------------------- 1/2 [google-genai]
   -------------------- ------------------- 1/2 [google-genai]
   -------------------- ------------------- 1/2 [google-genai]
   -------------------- ------------------- 1/2 [google-genai]
   ---------------------------------------- 2/2 [google-genai]

Note: you may need to restart the kernel to use updated packages.


In [26]:
from google import genai
import time

client = genai.Client()

myfile = client.files.upload(file="../../data/raw/videos/9JzNU0hrR0E.mp4")
print(f"업로드 완료: {myfile.name}")

print("서버 처리 대기")
while myfile.state.name == "PROCESSING":
    print(".", end="", flush=True) # 진행상황 표시
    time.sleep(5) # 5초마다 확인
    # 상태 정보를 서버에서 다시 받아와서 갱신
    myfile = client.files.get(name=myfile.name)

if myfile.state.name == "FAILED":
    raise ValueError("파일 처리 실패 (State: FAILED)")

print(f"\n처리 완료 (State: {myfile.state.name})")


response = client.models.generate_content(
    model="gemini-2.5-flash", 
    contents=[myfile, "이 비디오의 음성, 동작, 자막, 배경음악을 종합적으로 해석해서 시간 순서대로 텍스트로 정리해줘."]
)

print("\n=== 분석 결과 ===")
print(response.text)

업로드 완료: files/1z53rvja08sf
서버 처리 대기
.
처리 완료 (State: ACTIVE)

=== 분석 결과 ===
물론입니다! 이 비디오의 시간 순서에 따라 정보를 정리해 드리겠습니다.

[00:00] 영상이 시작하며, '골반통신'이라는 노란색 글씨가 검은색 배경에 나타납니다. 신나는 배경음악이 시작됩니다.
[00:01] 한 여성이 공원처럼 보이는 야외에서 휴대폰을 들고 서 있습니다. 그녀의 오른쪽 위에서 파란색 말풍선이 나타나며 "자기야 나 공원 도착"이라는 메시지가 보입니다.
[00:02] 노란색 말풍선이 세 개 나타나며 "헤어지자", "지호야", "쪽팔려서 너랑"이라는 메시지가 추가됩니다.
[00:04] 또 다른 노란색 말풍선이 나타나며 "못 다니겠어"라는 메시지가 보입니다. 이어서 파란색 말풍선이 나타나며 "알겠어.."라는 메시지가 보입니다.
[00:07] 여성은 고개를 푹 숙인 채 우울한 표정을 짓습니다. 그녀의 머리 위에서 말풍선이 나타나며 "내 골반이 멈추지 않는 탓일까? ㅠ.ㅠ"라는 메시지가 보입니다.
[00:12] 노란색 말풍선이 나타나며 "같이 놀이공원 가려고 했는데"라는 메시지가 보입니다.
[00:14] 파란색 말풍선이 나타나며 "혼자 가야겠다.."라는 메시지가 보입니다.
[00:15] 배경이 놀이공원으로 바뀝니다. 여성은 여전히 우울한 표정으로 서 있습니다.
[00:16] 흰색 말풍선이 두 개 나타나며 왼쪽에서는 "엄마 저 사람 이상해", 오른쪽에서는 "그런 거 보는 거 아니야"라는 메시지가 보입니다.
[00:19] 구름 모양의 말풍선이 나타나며 "ㅠㅠ 나 없이 어떻게 살지?"라는 메시지가 보입니다.
[00:22] 구름 모양의 말풍선이 다시 나타나며 "놀이기구라도 타서 슬픔을 잊어야겠다"라는 메시지가 보입니다.
[00:24] 여성은 놀이기구를 타러 계단을 올라갑니다.
[00:26] 놀이기구에 탑승하려는 여성을 클로즈업한 화면에서 흰색 말풍선이 나타나며 "골반 때문에 앉을 수가 없어요"라는 메시지가 보입니다.
[00:28

### 유튜브 url로 분석

##### Gemini 2.5-Flash (API로 유튜브url 보내기)

In [ ]:
from google import genai
from google.genai import types
import os

# 1. 클라이언트 설정
client = genai.Client()

# 2. 모델 및 프롬프트 설정 (다운로드 과정 삭제)
response = client.models.generate_content(
    model='gemini-2.5-flash',  # 문서에 나온 최신 모델
    contents=types.Content(
        parts=[
            # 유튜브 링크를 직접 넣습니다.
            types.Part(
                file_data=types.FileData(file_uri='https://www.youtube.com/watch?v=9JzNU0hrR0E')
            ),
            types.Part(text='이 비디오의 음성, 동작, 자막, 배경음악을 종합적으로 해석해서 시간 순서대로 텍스트로 정리해줘.')
        ]
    )
)

print(response.text)

Sure, here's a detailed log of the video's events:

(Video begins)
[00:00] A young woman is standing in a park, looking at her phone.
[00:01] A blue speech bubble appears on the right side of the screen with the text, "자기야 나 공원 도착" (Honey, I've arrived at the park).
[00:02] A yellow speech bubble appears on the left side of the screen with the text, "헤어지자" (Let's break up).
[00:03] Another yellow speech bubble appears below the previous one with the text, "지호야" (Jiho).
[00:04] A third yellow speech bubble appears with the text, "쪽팔려서 너랑 못 다니겠어" (It's embarrassing to be seen with you). A blue speech bubble on the right says, "알겠어.." (Okay..). The woman looks shocked and disappointed.
[00:07] The woman stands still, then a speech bubble appears from her with the text, "내 골반이 멈추지 않는 탓일까? ㅠㅠ" (Is it because my pelvis won't stop moving? ㅠㅠ).
[00:12] Another speech bubble appears from her with the text, "같이 놀이공원 가려고 했는데" (I was going to go to the amusement park with him).
[00:14] A speech bu

##### 그냥 한 번 더 해보기

In [ ]:
from google import genai
from google.genai import types
import os

# 1. 클라이언트 설정
client = genai.Client()

# 2. 모델 및 프롬프트 설정 (다운로드 과정 삭제)
response = client.models.generate_content(
    model='gemini-2.5-flash',  # 문서에 나온 최신 모델
    contents=types.Content(
        parts=[
            # 유튜브 링크를 직접 넣습니다.
            types.Part(
                file_data=types.FileData(file_uri='https://www.youtube.com/watch?v=9JzNU0hrR0E')
            ),
            types.Part(text='이 비디오의 음성, 동작, 자막, 배경음악을 종합적으로 해석해서 시간 순서대로 텍스트로 정리해줘.')
        ]
    )
)

print(response.text)

다음은 동영상 "골반통신"의 음성, 동작, 자막, 배경음악을 시간 순서대로 정리한 내용입니다.

[배경음악: 경쾌하고 활기찬 인스트루멘탈 음악]

**00:00 - 00:01**
*   **자막:** 골반통신
*   **동작:** 여자가 공원 같은 야외에서 서서 휴대폰을 보고 있습니다.
*   **자막:** 자기야 나 공원 도착 (파란색 말풍선)

**00:02 - 00:06**
*   **동작:** 여자가 휴대폰 화면을 보며 놀란 표정을 짓습니다.
*   **자막:**
    *   헤어지자 (노란색 말풍선)
    *   지호야 (노란색 말풍선)
    *   쪽팔려서 너랑 못 다니겠어 (노란색 말풍선)
    *   알겠어.. (파란색 말풍선)

**00:07 - 00:11**
*   **동작:** 여자가 휴대폰을 내리고 슬픈 표정으로 서 있습니다. 몸이 좌우로 살짝 흔들립니다.
*   **자막:** 내 골반이 멈추지 않는 탓일까? (생각 말풍선)

**00:12 - 00:14**
*   **동작:** 여자가 위를 가리키며 생각하는 듯한 자세를 취합니다. 몸이 좌우로 살짝 흔들립니다.
*   **자막:**
    *   같이 놀이공원 가려고 했는데 (생각 말풍선)
    *   혼자 가야겠다.. (생각 말풍선)

**00:15 - 00:19**
*   **동작:** 장소가 놀이공원으로 바뀌었습니다. 여자가 팔을 벌리고 슬픈 표정으로 서 있습니다. 몸이 좌우로 살짝 흔들립니다.
*   **자막:**
    *   엄마 저 사람 이상해 (파란색 말풍선)
    *   그런 거 보는 거 아니야 (노란색 말풍선)

**00:20 - 00:23**
*   **동작:** 여자가 여전히 팔을 벌리고 서서 생각하는 듯한 표정을 짓습니다. 몸이 좌우로 살짝 흔들립니다.
*   **자막:**
    *   ㅠㅠㅠ 나 쟤 없이 어떻게 살지? (생각 말풍선)
    *   놀이기구라도 타서 슬픔을 잊어야겠다 (생각 말풍선)

**00:24 - 00:26**
*   **동작:*

##### Gemini 2.5-Flash-Lite (API로 유튜브url 보내기)

In [23]:
from google import genai
from google.genai import types
import os

# 1. 클라이언트 설정
client = genai.Client()

# 2. 모델 및 프롬프트 설정 (다운로드 과정 삭제)
response = client.models.generate_content(
    model='gemini-2.5-flash-lite',  # 문서에 나온 최신 모델
    contents=types.Content(
        parts=[
            # 유튜브 링크를 직접 넣습니다.
            types.Part(
                file_data=types.FileData(file_uri='https://www.youtube.com/watch?v=9JzNU0hrR0E')
            ),
            types.Part(text='이 비디오의 음성, 동작, 자막, 배경음악을 종합적으로 해석해서 시간 순서대로 텍스트로 정리해줘.')
        ]
    )
)

print(response.text)

## 영상 내용 분석

이 비디오는 가상의 연인 간의 대화를 보여주는 콩트 형식의 영상입니다. 영상은 한 여성이 공원에서 휴대폰을 보고 있는 장면으로 시작하며, 여러 가지 상황과 대화가 이어진 후 결국 아쿠아리움으로 마무리됩니다.

### 타임라인별 내용 정리

**00:00 - 00:01: 오프닝**
* **음성:** "골반 통신"이라는 제목이 등장합니다.
* **동작:** 없음
* **자막:** 골반 통신
* **배경음악:** 없음

**00:01 - 00:05: 공원에서의 대화 1**
* **음성:** "자기야 나 공원 도착" (여성의 휴대폰에서 흘러나오는 소리)
* **동작:** 여성이 휴대폰을 보며 미소 짓습니다.
* **자막:**
    * 자기야 나 공원 도착
* **배경음악:** 잔잔한 BGM

**00:05 - 00:11: 공원에서의 대화 2**
* **음성:**
    * 혜어지자
    * 지호야
    * 톡팔려서 너랑
    * 못 다니겠어
* **동작:** 여성의 표정이 미묘하게 변합니다.
* **자막:**
    * 혜어지자
    * 지호야
    * 톡팔려서 너랑
    * 못 다니겠어
* **배경음악:** 잔잔한 BGM

**00:11 - 00:15: 공원에서의 대화 3**
* **음성:**
    * 알겠어..
    * 내 골반이 멈추지 않는 탓일까?
    * ㅠㅠ
    * 같이 놀이공원
    * 가려고 했는데
    * 혼자 가야겠다..
* **동작:** 여성은 휴대폰을 내려놓고 좌절하는 표정을 짓습니다.
* **자막:**
    * 알겠어..
    * 내 골반이 멈추지 않는 탓일까?
    * ㅠㅠ
    * 같이 놀이공원
    * 가려고 했는데
    * 혼자 가야겠다..
* **배경음악:** 잔잔한 BGM

**00:15 - 00:25: 놀이공원 장면**
* **음성:**
    * 엄마 저 사람 이상해
    * 그런 거 보는 거 아니야
    * ㅠㅠ
    * 나 개 없이
    * 어떻게 살지?
   

##### Gemini 3-Flash (API로 유튜브url 보내기)

In [25]:
from google import genai
from google.genai import types
import os

# 1. 클라이언트 설정
client = genai.Client()

# 2. 모델 및 프롬프트 설정 (다운로드 과정 삭제)
response = client.models.generate_content(
    model='gemini-3-flash-preview',  # 문서에 나온 최신 모델
    contents=types.Content(
        parts=[
            # 유튜브 링크를 직접 넣습니다.
            types.Part(
                file_data=types.FileData(file_uri='https://www.youtube.com/watch?v=9JzNU0hrR0E')
            ),
            types.Part(text='이 비디오의 음성, 동작, 자막, 배경음악을 종합적으로 해석해서 시간 순서대로 텍스트로 정리해줘.')
        ]
    )
)

print(response.text)

다음은 영상의 음성, 동작, 자막, 배경음악을 바탕으로 시간 순서대로 정리한 내용입니다.

**00:00 - 00:05: 공원에서의 이별**
*   **상황 및 자막:** 한 여성이 공원에서 핸드폰을 보며 남자친구(지호)에게 "자기야 나 공원 도착"이라는 메시지를 보냅니다. 하지만 곧바로 "헤어지자", "지호야 쪽팔려서 너랑 못 다니겠어"라는 이별 통보를 받습니다. 여성은 체념한 듯 "알겠어.."라고 답합니다.
*   **동작:** 여성은 슬픈 표정으로 핸드폰을 내려다봅니다.
*   **배경음악:** 경쾌하고 발랄한 비트의 음악이 시작됩니다.

**00:05 - 00:15: 멈추지 않는 골반 댄스**
*   **자막:** "내 골반이 멈추지 않는 탓일까?", "같이 놀이공원 가려고 했는데", "혼자 가야겠다.."라는 자막이 등장합니다.
*   **동작:** 여성은 자신의 의지와 상관없이 골반이 좌우로 계속 흔들리는 듯한 춤을 추기 시작합니다. 이별의 슬픔과 대조되는 경쾌한 움직임입니다.

**00:15 - 00:23: 놀이공원 도착과 사람들의 시선**
*   **상황 및 자막:** 장면이 놀이공원으로 바뀝니다. 여성이 계속 춤을 추자 주변 아이가 "엄마 저 사람 이상해"라고 말하고, 엄마는 "그런 거 보는 거 아니야"라며 기피합니다. 여성은 "나 얘 없이 어떻게 살지?", "놀이기구라도 타서 슬픔을 잊어야겠다"고 생각합니다.
*   **동작:** 여성은 놀이공원 한복판에서 양팔을 벌리고 계속해서 골반을 흔듭니다.

**00:23 - 00:29: 놀이기구 탑승 거부**
*   **상황 및 자막:** 놀이기구에 앉으려 하지만, "골반 때문에 앉을 수가 없어요"라고 말합니다. 직원은 "그럼 못 타세요 내려가세요"라고 답하고 여성은 "네..."라며 내립니다.
*   **동작:** 놀이기구 좌석에 앉으려 시도하지만 멈추지 않는 골반 때문에 결국 포기합니다.

**00:29 - 00:39: 낯선 여성과의 만남**
*   **상황 및 자막:** "놀이기구한테도 까였네"라며